# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-6-luna'
openai = OpenAI()

API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/11/11/ai-sdlc-the-ai-native-software-engineer-in-the-agentic-era/',
 'https://edwarddonner.com/2026/11/11/ai-sdlc-the-ai-native-software-engineer-in-the-agentic-era/',
 'https://edwarddonner.com/2026/10/14/ai-fde-the-forward-deployed-engineer-agentic-ai-architect/',
 'https://edwarddonner.com/2026/10/14/ai-fde-the-forward

## First step: Have GPT-6 Luna figure out which links are relevant

### Use a call to GPT-6 Luna to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [36]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About pages, or a Company pages, or Careers/Jobs pages, Post pages, Blog pages, etc...
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [37]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [38]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/11/11/ai-sdlc-the-ai-native-software-engineer-in-the-agentic-era/
https://edwarddo

In [39]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        reasoning_effort="none",
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [40]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'posts page', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'company website',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'company announcement',
   'url': 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html'},
  {'type': 'blog post',
   'url': 'https://edwarddonner.com/2026/11/11/ai-sdlc-the-ai-native-software-engineer-in-the-agentic-era/'},
  {'type': 'blog post',
   'url': 'https://edwarddonner.com/2026/10/14/ai-fde-the-forward-deployed-engineer-agentic-ai-architect/'},
  {'type': 'blog post',
   'url': 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/'},
  {'type': 'blog post',
   'url': 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/'},
  {'type': 'LinkedIn profile', 'url': 'https://www.link

In [41]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        reasoning_effort="none",
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [42]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-6-luna
Found 9 relevant links


{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'company website',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'blog', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'blog post',
   'url': 'https://edwarddonner.com/2026/11/11/ai-sdlc-the-ai-native-software-engineer-in-the-agentic-era/'},
  {'type': 'blog post',
   'url': 'https://edwarddonner.com/2026/10/14/ai-fde-the-forward-deployed-engineer-agentic-ai-architect/'},
  {'type': 'blog post',
   'url': 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/'},
  {'type': 'blog post',
   'url': 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/'},
  {'type': 'LinkedIn profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'social media profile', 'url': 'https://twitter.com/edwarddonner'}]}

In [43]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-6-luna
Found 9 relevant links


{'links': [{'type': 'company homepage', 'url': 'https://huggingface.co/'},
  {'type': 'enterprise', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing', 'url': 'https://huggingface.co/pricing'},
  {'type': 'company blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'careers', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'company on GitHub', 'url': 'https://github.com/huggingface'},
  {'type': 'company on LinkedIn',
   'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'company on X', 'url': 'https://twitter.com/huggingface'},
  {'type': 'company status', 'url': 'https://status.huggingface.co/'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-6 Luna

In [ ]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [45]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-6-luna
Found 8 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Search models, datasets, users...
Ctrl K
Models
Datasets
Spaces
Buckets
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Hardware
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
xtools-at/DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic
Updated
3 days ago
•
27k
•
7.47k
MiniMaxAI/MiniMax-H3-Image
Updated
4 days ago
•
571k
•
1.15k
Qwen/Qwen3.8-27B
Updated
8 days ago
•
6.55M
•
20.1k
Lightricks/LTX-2.5
Updated
Oct 11
•
2.75M
•
12k

In [89]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [85]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a sales brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    # user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [77]:
print(get_brochure_user_prompt("HuggingFace", "https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-6-luna
Found 18 relevant links

You are looking at a company called: HuggingFace
Here are the contents of its landing page and other relevant pages;
use this information to build a sales brochure of the company in markdown without code blocks.


## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Search models, datasets, users...
Ctrl K
Models
Datasets
Spaces
Buckets
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Hardware
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
xtools-at/DeepSeek-V4-Pro-Qwen3.5

In [78]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-6-luna",
        reasoning_effort="none",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [79]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-6-luna
Found 10 relevant links


# Hugging Face
## The AI community building the future

Hugging Face is a collaboration platform for the machine learning community. It brings together tools, infrastructure, and a global community so people can discover, build, share, and run AI.

From a first experiment to an enterprise deployment, Hugging Face helps teams move from exploring machine learning to putting it to work.

## One platform for the AI lifecycle

Explore a broad ecosystem of open machine learning resources:

- **Models:** Browse more than 2 million models across AI tasks and communities.
- **Datasets:** Discover and share more than 500,000 datasets.
- **Spaces:** Try and publish interactive AI applications—over 1 million are available to explore.
- **Collaboration:** Host projects and work with the ML community through shared tools and resources.
- **Inference and deployment:** Access Inference Providers, deploy with Inference Endpoints, and use Storage Buckets.
- **Learning and community:** Find documentation, courses, discussions, research, and practical examples.

Whether you’re evaluating a model, building an application, or sharing research, the Hub provides a place to get started and collaborate.

## Built for teams and enterprises

Hugging Face offers Team and Enterprise plans, Hugging Face PRO, and Enterprise Support. Its platform and services are designed to help organizations work with machine learning resources and build AI applications, with options spanning model discovery, collaboration, inference, deployment, and storage.

**For prospective customers:** Explore the platform to find models, datasets, and applications relevant to your work, then consider the available plans and services for your team’s needs.

## An active, open community

Hugging Face describes its mission as helping democratize AI and build an open, ethical AI future. Its community brings together engineers, researchers, creators, and learners who share work and learn from one another.

The community is visible across the Hub, forums, blog, GitHub, Discord, and other channels. Discussions range from beginner questions and project showcases to research on agents, model evaluation, robotics, and AI safety. Community articles and collaborations feature organizations including NVIDIA, Salesforce, AllenAI, Figma, and Google AI Edge.

## Company at a glance

- **Founded:** 2016
- **Industry:** Software development
- **Company type:** Privately held
- **Listed company size:** 51–200 employees
- **Locations highlighted:** Paris and New York City, with a global community
- **Focus areas:** Machine learning, natural language processing, and deep learning

## Careers

Hugging Face invites prospective candidates to explore its current openings. The company’s work sits at the intersection of open-source software, machine learning, research, and community collaboration—an opportunity to contribute to tools and infrastructure used by people building AI.

**Interested in joining?** Visit the Hugging Face careers page for current roles.

## Explore Hugging Face

- Browse models, datasets, and AI applications on the Hugging Face Hub
- Learn through documentation, courses, and community articles
- Join discussions in the Hugging Face forums
- Explore Team & Enterprise plans and platform services
- View current career openings

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [90]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-6-luna",
        reasoning_effort="none",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [91]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-6-luna
Found 9 relevant links


# Hugging Face  
## The AI community building the future

Hugging Face is a collaboration platform for the machine-learning community—helping people and organizations discover, build, share, and run AI. Its mission is to help solve and democratize artificial intelligence, and to support an open, collaborative path toward the future of AI.

**Explore a large ecosystem of models, datasets, and applications—all in one place.**

## One platform for the machine-learning lifecycle

- **Models:** Browse more than **2 million models** shared by the community.
- **Datasets:** Discover more than **500,000 datasets** for training, evaluation, and experimentation.
- **Spaces:** Explore more than **1 million applications** and interactive AI demos.
- **Collaboration:** Publish work, learn from others, and build together through a central hub for machine learning.
- **Deployment and infrastructure:** Use services including Inference Providers, Inference Endpoints, and Storage Buckets.
- **Tools and learning:** Find documentation, community articles, discussions, and open-source resources.

From testing a model in an interactive application to contributing datasets and research, Hugging Face brings practical tools and community knowledge together.

## Built for organizations, too

Hugging Face offers **Team and Enterprise plans**, along with **Enterprise Support**, for organizations adopting machine learning. Teams can explore the platform’s model and data ecosystem and use hosted inference and storage options as part of their workflows.

## A community of builders

The Hub brings together machine-learning engineers, researchers, developers, organizations, and AI enthusiasts. Community contributions span natural language processing, audio, computer vision, robotics, reinforcement learning, and more. The company’s blog features community research, practical guides, open-source collaborations, partnerships, and case studies.

Hugging Face describes its goal as empowering the next generation of machine-learning engineers, scientists, and end users to learn, collaborate, and share work in support of an open and ethical AI future.

## Company at a glance

- **Founded:** 2016
- **Industry:** Software development
- **Locations:** Paris and New York City, with a global community
- **Company type:** Privately held
- **Focus areas:** Machine learning, natural language processing, and deep learning

## Careers

Interested in helping shape the future of open and collaborative AI? Hugging Face shares current openings through its careers page. Its public profiles describe a distributed presence across Paris, New York City, and beyond. Explore current roles directly on the **Hugging Face careers page**.

## Get started

Discover models, datasets, and AI applications—or explore Team and Enterprise options to learn how Hugging Face can support your organization’s machine-learning work.

In [88]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-6-luna
Found 12 relevant links


# 🤗 Hugging Face
## The AI community building the future—without making you build everything from scratch.

**Models, datasets, apps, tools, and a very large community, all in one place.** Hugging Face is the collaboration platform for machine-learning builders: discover what’s out there, share what you’ve made, and turn promising experiments into useful AI.

### Your AI workbench, with room to grow

- **Models:** Explore millions of models across text, image, audio, video, and more.
- **Datasets:** Find and share data for a wide range of machine-learning tasks.
- **Spaces:** Build and try interactive AI applications—because a model is more fun when you can actually use it.
- **Open-source tools:** Use libraries including Transformers, Diffusers, Datasets, PEFT, and more.
- **Learn:** Start with courses in LLMs, agents, robotics, audio, computer vision, and other topics. There’s even an “a smol course”—proof that AI education can have a sense of scale.

### From “interesting model” to “working product”

Browse and test models, build demos, fine-tune systems, and deploy through inference providers or dedicated, managed Inference Endpoints. Hugging Face also supports teams and enterprises with private collaboration, storage, and enterprise support.

Whether you’re exploring an idea or deploying AI for real users, you can get from discovery to deployment without first assembling your own universe of tools.

### Built with the community

Hugging Face describes itself as a platform for people to **share, explore, discover, and experiment with open-source machine learning**. Its community includes researchers, developers, organizations, and curious builders working across the AI ecosystem. The Hub is busy: models, datasets, apps, articles, papers, and projects are being updated and shared every day.

The company’s stated ambition is to help democratize AI and build an **open and ethical AI future**. The practical version: make useful tools and knowledge easier for more people to access—and make collaboration part of the process.

### For teams with serious plans (and sensible questions)

Use the platform to explore a broad catalog of models, connect to inference partners, and deploy models on managed infrastructure. Teams and enterprises can also look into enterprise plans and support.

In short: bring your AI project. Hugging Face brings a community, tools, and several million things to investigate before lunch.

### A place to learn—and work

Hugging Face was founded in **2016** and operates in **New York and Paris**, with a global community. Its learning resources range from large language models and agents to robotics and deep reinforcement learning.

Interested in joining the mission? Hugging Face has a careers page with current openings. For people who like open-source software, machine learning, and turning “what if?” into something others can use, that’s a promising combination.

## Come build what’s next 🤗

**Explore models. Share your work. Find your people.**  
And yes, you can start by browsing. The rabbit hole is well documented.

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>